In [4]:
import os
# 缓解显存碎片和过度预留的问题
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [5]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from tqdm import tqdm
import openai
import json
import time
import os
from PIL import Image

In [6]:
import torch,json,sys,os,time,re,transformers
from tqdm import tqdm

# 检查GPU是否可用
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("Current CUDA device:", torch.cuda.current_device())
print("Torch version:", torch.__version__)

print(sys.version)

torch.cuda.empty_cache()

CUDA available: True
CUDA device count: 4
Current CUDA device: 0
Torch version: 2.6.0+cu124
3.11.14 | packaged by conda-forge | (main, Oct 22 2025, 22:46:25) [GCC 14.3.0]


In [7]:
# 检查当前notebook使用的Python环境
print("Python executable:", sys.executable)
print("Python version:", sys.version)

# 检查transformers版本和位置
print("\nTransformers version:", transformers.__version__)
print("Transformers location:", transformers.__file__)

Python executable: /root/miniconda3/envs/py311/bin/python
Python version: 3.11.14 | packaged by conda-forge | (main, Oct 22 2025, 22:46:25) [GCC 14.3.0]

Transformers version: 4.46.3
Transformers location: /root/miniconda3/envs/py311/lib/python3.11/site-packages/transformers/__init__.py


## 加载模型

In [8]:
from transformers import AutoModel, AutoTokenizer

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
ocr_model_path = "../model/deepseek-ocr"

tokenizer = AutoTokenizer.from_pretrained(ocr_model_path, _attn_implementation='flash_attention_2', trust_remote_code=True)
model = AutoModel.from_pretrained(
    ocr_model_path, trust_remote_code=True, use_safetensors=True
)
model = model.eval().cuda("cuda:1").to(torch.bfloat16)

# prompt = "<image>\nFree OCR. "
# prompt = "<image>\n<|grounding|>Convert the document to markdown. "
# image_file = 'your_image.jpg'
# output_path = 'your/output/dir'

# infer(self, tokenizer, prompt='', image_file='', output_path = ' ', base_size = 1024, image_size = 640, crop_mode = True, test_compress = False, save_results = False):

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 640, image_size = 640, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# Gundam: base_size = 1024, image_size = 640, crop_mode = True

# res = model.infer(tokenizer, prompt=prompt, image_file=image_file, output_path = output_path, base_size = 1024, image_size = 640, crop_mode=True, save_results = True, test_compress = True)


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at ../model/deepseek-ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 提示词

In [9]:
# 论文中使用的prompt
prompt = "<image>\nFree OCR. "
# 官方仓库实例的prompt
# prompt = "<image>\n<|grounding|>Convert the document to markdown. "  

## 测试函数

In [ ]:
# 原论文中只评估了 tiny 和 small 模型 分别对应的image_size base_size = 512 640
def ocr_image(tokenizer, model, image_file, output_path, image_size=512,base_size=512):
    res = model.infer(
        tokenizer=tokenizer,
        prompt=prompt,
        image_file=image_file,
        output_path=output_path,
        base_size=base_size,
        image_size=image_size,
        # crop_mode=True,
        crop_mode=False,
        # save_results=True,    # 这个设置会将结果保存到output_path目录下
        save_results=False,
        eval_mode=True,         # 评估模式，不保存结果，将结果返回
        test_compress=True,
    )
    return res

def ocr_image_no_compress(tokenizer, model, image_file, output_path, image_size=512,base_size=512):
    """
    不使用压缩的 OCR 结果, 作为对比实验
    """
    res = model.infer(
        tokenizer=tokenizer,
        prompt=prompt,
        image_file=image_file,
        output_path=output_path,
        base_size=base_size,
        image_size=image_size,
        # crop_mode=True,
        crop_mode=False,
        # save_results=True,    # 这个设置会将结果保存到output_path目录下
        save_results=False,
        eval_mode=True,         # 评估模式，不保存结果，将结果返回
        test_compress=False,
    )
    return res

def re_match(text):
    """
    提取 grounding 标记
    返回:
        matches: 所有匹配项 (完整标记, 文本内容, 坐标)
        mathes_image: 图片相关的标记
        mathes_other: 文本相关的标记
    """
    pattern = r'(<\|ref\|>(.*?)<\|/ref\|><\|det\|>(.*?)<\|/det\|>)'
    matches = re.findall(pattern, text, re.DOTALL)

    mathes_image = []
    mathes_other = []
    for a_match in matches:
        if '<|ref|>image<|/ref|>' in a_match[0]:
            mathes_image.append(a_match[0])
        else:
            mathes_other.append(a_match[0])
    return matches, mathes_image, mathes_other

def clean_ocr_output(text):
    """
    官方的清理方法：
    1. 提取所有标记
    2. 替换图片标记为 markdown 图片格式
    3. 删除所有文本标记
    """
    matches_ref, matches_images, mathes_other = re_match(text)
    
    # 替换图片标记
    for idx, a_match_image in enumerate(matches_images):
        text = text.replace(a_match_image, f'![](images/{idx}.jpg)\n')
    
    # 删除所有文本标记
    for idx, a_match_other in enumerate(mathes_other):
        text = text.replace(a_match_other, '')
    
    # 额外清理
    text = text.replace('\\coloneqq', ':=').replace('\\eqqcolon', '=:')
    text = text.replace('\n\n\n\n', '\n\n').replace('\n\n\n', '\n\n')
    text = text.replace('<center>', '').replace('</center>', '')
    
    return text.strip()

def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

def process_single_image(tokenizer, model, image_name, output_path, imgs_dir, mode):
    """
    处理单张图片，返回清理后的 OCR 文本
    """
    if mode == "tiny":
        IMAGE_SIZE = 512
        BASE_SIZE = 512
        process_func = ocr_image
    elif mode == "small":
        IMAGE_SIZE = 640
        BASE_SIZE = 640
        process_func = ocr_image
    elif mode == "raw":
        img = Image.open(os.path.join(imgs_dir, image_name))
        w, h = img.size
        long_side = max(w, h)
        IMAGE_SIZE = long_side
        BASE_SIZE = long_side
        process_func = ocr_image_no_compress
        
    image_path = os.path.join(imgs_dir, image_name)
    res = process_func(
        tokenizer,
        model,
        image_path,
        output_path=output_path,
        image_size=IMAGE_SIZE,
        base_size=BASE_SIZE,
        )
    clean_text = clean_ocr_output(res)
    return clean_text

def ocr(tokenizer, model, data_path=None, output_path = "../output", save_path=None, imgs_dir=None, mode="tiny"):
    ocr_results = []
    image_names = [f"en_{i+1}.png" for i in range(112)]
    # image_paths = [os.path.join(images_dir, img_name) for img_name in image_names]
    print(f"开始处理 {len(image_names)} 张图片...")
    
    for image_name in tqdm(image_names):
        # TODO
        torch.cuda.empty_cache()        # 释放空间
        clean_text = process_single_image(tokenizer, model, image_name, output_path, imgs_dir, mode)
        ocr_results.append({
            "image": image_name,
            "ocr_text": clean_text
        })
    # 按照image name重新排序
    ocr_results = sorted(
        ocr_results,
        key=lambda x: int(x["image"].split("_")[-1].split(".")[0])
    )
    
    # 将结果合并到原始数据中
    data = load_data(data_path)
    for item in data:
        image_name = item["image"]
        for ocr_item in ocr_results:
            if ocr_item["image"] == image_name:
                item["ocr_text"] = ocr_item["ocr_text"]
                break
    
    # 将最终结果保存到文件中
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    
    
    print(f"\n结果已保存到: {save_path}")

## OCR

In [12]:
data_path = "../fox_data/data.json"

In [ ]:
# ocr(tokenizer, model, data_path, "../output", save_path="../results/ocr/en_png_small.json", imgs_dir="../fox_data/en_png", mode="small")

开始处理 112 张图片...


  0%|          | 0/112 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 100, 1280])
NO PATCHES


  1%|          | 1/112 [00:52<1:36:50, 52.34s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 100, 1280])
NO PATCHES


...

directly resize
BASE:  torch.Size([1, 100, 1280])
NO PATCHES


 99%|█████████▉| 111/112 [1:27:32<00:45, 45.09s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 100, 1280])
NO PATCHES


100%|██████████| 112/112 [1:28:04<00:00, 47.18s/it]


结果已保存到: ../results/ocr/en_png_small.json


In [ ]:
# ocr(tokenizer, model, data_path, "../output", save_path="../results/ocr/distort_small.json", imgs_dir="../fox_data/distort", mode="small")

开始处理 112 张图片...


  0%|          | 0/112 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 100, 1280])
NO PATCHES


  1%|          | 1/112 [01:04<1:58:48, 64.22s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 100, 1280])
NO PATCHES


...

directly resize
BASE:  torch.Size([1, 100, 1280])
NO PATCHES


 99%|█████████▉| 111/112 [1:40:28<00:49, 49.72s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 100, 1280])
NO PATCHES


100%|██████████| 112/112 [1:41:08<00:00, 54.19s/it]


结果已保存到: ../results/ocr/distort_small.json


In [ ]:
# ocr(tokenizer, model, data_path, "../output", save_path="../results/ocr/from_text_small.json", imgs_dir="../fox_data/from_text", mode="small")

开始处理 112 张图片...


  0%|          | 0/112 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 100, 1280])
NO PATCHES


  1%|          | 1/112 [00:55<1:43:24, 55.90s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 100, 1280])
NO PATCHES


...

directly resize
BASE:  torch.Size([1, 100, 1280])
NO PATCHES


 99%|█████████▉| 111/112 [1:41:16<00:44, 44.51s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


directly resize
BASE:  torch.Size([1, 100, 1280])
NO PATCHES


100%|██████████| 112/112 [1:41:51<00:00, 54.57s/it]


结果已保存到: ../results/ocr/from_text_small.json


In [13]:
# ocr(tokenizer, model, data_path, "../output", save_path="../results/ocr/distort_raw.json", imgs_dir="../fox_data/distort", mode="raw")

开始处理 112 张图片...


  0%|          | 0/112 [00:00<?, ?it/s]/root/miniconda3/envs/py311/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
The attention layers in this model are 

BASE:  torch.Size([1, 196, 1280])
NO PATCHES


  1%|          | 1/112 [02:05<3:52:18, 125.57s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 324, 1280])
NO PATCHES


...

BASE:  torch.Size([1, 196, 1280])
NO PATCHES


 99%|█████████▉| 111/112 [7:11:46<03:45, 225.27s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 196, 1280])
NO PATCHES


100%|██████████| 112/112 [7:13:25<00:00, 232.19s/it]


结果已保存到: ../results/ocr/distort_raw.json
